# TP 1 – Image Classification with Artificial Neural Networks
## Dataset: Food (3 classes)

In this practical, you will build and train an ANN to classify food images using the Food-11 dataset.

We work with **3 classes only**: `Bread`, `Dairy product`, `Egg`.

The dataset is already split into three folders:
- `training/` — used to train the model
- `validation/` — used to monitor the model during training
- `test/` — used to test the final model

**Before you start:** download the Food dataset and place it in the same folder as this notebook. Your directory should look like this:
```
FoodDataSet/
    training/
        Bread/
        Dairy product/
        Egg/
    validation/
        Bread/
        Dairy product/
        Egg/
    evaluation/
        Bread/
        Dairy product/
        Egg/
```

---
## Step 1 — Import Libraries

We import the tools we need:
- `numpy` and `matplotlib` for data handling and visualization
- `tensorflow.keras` to build and train the neural network
- `sklearn` for the confusion matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print("Libraries imported successfully.")

---
## Step 2 — Load the Dataset

`ImageDataGenerator` loads images directly from folders and applies **rescaling** (dividing pixel values by 255) to normalize them to the range [0, 1]. This is important because neural networks train better on small, consistent input values.

`flow_from_directory` automatically assigns labels based on subfolder names and resizes all images to `64x64` pixels.

In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen   = ImageDataGenerator(rescale=1./255)
test_datagen  = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    r"C:\Users\lenovo\Desktop\FoodDataSet\train",
    target_size=(64, 64),
    batch_size=32,
    class_mode="categorical"
)

val_generator = val_datagen.flow_from_directory(
    r"C:\Users\lenovo\Desktop\FoodDataSet\validation",
    target_size=(64, 64),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    r"C:\Users\lenovo\Desktop\FoodDataSet\test",
    target_size=(64, 64),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

print("\nClass mapping:", train_generator.class_indices)

---
## Step 3 — Visualize Sample Images

Before training, always look at your data. This helps you understand what the model will learn from.

In [ ]:
images, labels = next(train_generator)
class_names = list(train_generator.class_indices.keys())

plt.figure(figsize=(12, 4))
for i in range(8):
    plt.subplot(2, 4, i+1)
    plt.imshow(images[i])
    plt.title(class_names[np.argmax(labels[i])])
    plt.axis('off')
plt.suptitle('Sample training images', fontsize=13)
plt.tight_layout()
plt.show()

---
## Step 4 — Build the ANN Model

An ANN expects a **1D input vector**, but images are 3D arrays (height × width × channels). The `Flatten` layer converts each 64×64×3 image into a single vector of 12,288 values.

The network then has two hidden layers with **ReLU** activation, and an output layer with **Softmax** which converts raw scores into probabilities (one per class).

**Your task:** once you get a first result, come back here and try changing:
- the number of neurons in the hidden layers (e.g. 256, 64, 32)
- the number of hidden layers (add or remove a `Dense` layer)
- the activation function (`relu` → `tanh` or `sigmoid`)

Observe how each change affects the training curves and final accuracy.

In [ ]:
num_classes = train_generator.num_classes

model = Sequential([
    Flatten(input_shape=(64, 64, 3)),
    Dense(128, activation="relu"),   # try: 256, 64, 32
    Dense(64,  activation="relu"),   # try: removing this layer, or adding one more
    Dense(num_classes, activation="softmax")
])

model.summary()

---
## Step 5 — Compile the Model

Before training, we need to configure the learning process:
- **optimizer** (`adam`): the algorithm that updates the weights. Adam is adaptive and works well in most cases.
- **loss** (`categorical_crossentropy`): the function the model minimizes. Used for multi-class classification with one-hot labels.
- **metrics** (`accuracy`): what we track during training.

**Try:** change the learning rate of Adam (default is 0.001). Use `optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001)` and see if convergence is slower.

In [ ]:
model.compile(
    optimizer="adam",                   
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled.")

---
## Step 6 — Train the Model

We train for 10 epochs. One **epoch** = one full pass over the training data.

The validation set is evaluated after each epoch — it is never used for updating weights, only for monitoring the model's performance on unseen data.

**Try:** increase `epochs` to 20 or 30 and observe whether the model keeps improving or starts overfitting (val_loss increases while train_loss keeps decreasing).

In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10              # try: 20, 30
)

---
## Step 7 — Plot Training History

These curves show how the model evolved during training.

- **Loss** should decrease — if validation loss starts rising while training loss falls, the model is overfitting.
- **Accuracy** should increase — a large gap between train and validation accuracy also indicates overfitting.

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"],     label="Train")
plt.plot(history.history["val_accuracy"], label="Validation", linestyle="--")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"],     label="Train")
plt.plot(history.history["val_loss"], label="Validation", linestyle="--")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)

plt.suptitle("Training History", fontsize=13)
plt.tight_layout()
plt.show()

---
## Step 8 — Evaluate on the Test Set

The test set (evaluation folder) was never seen during training or validation. This gives an honest measure of how the model performs on truly new data.

In [ ]:
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)")

---
## Step 9 — Confusion Matrix

The confusion matrix shows which classes the model gets right and which ones it confuses with each other. The diagonal contains correct predictions — everything off the diagonal is a mistake.

In [ ]:
predictions = model.predict(test_generator)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes
labels = list(test_generator.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()

---
## Step 10 — Classification Report 

The classification report provides a detailed evaluation of the model’s performance for each class.

It includes:


- **Precision**: how many predicted samples of a class are correct.
- **Recall**: how many actual samples of a class were correctly detected.
- **F1-score**: balance between precision and recall.
- **Support**: number of true samples for each class.

This report helps us understand which classes the model predicts well and where it makes mistakes, beyond overall accuracy.

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred, target_names=labels))

---
## Hyperparameter Experiments

Go back to Steps 4, 5 and 6 and try the following changes one at a time. Each time, re-run all cells from Step 4 onwards and note the results in the table below.

| Experiment | Change | Train Acc | Val Acc | Test Acc | Observation |
|------------|--------|-----------|---------|----------|-------------|
| Baseline   | Default (128→64, adam, 10 epochs) |57% |55% |52.6% |acceptable performance Noticeable gap between training and validation accuracy → slight overfitting |
| Exp 1      | Neurons: 256→128 |56%|53.1% |51.6% |Accuracy decreased slightly → fewer neurons reduce learning capacity|
| Exp 2      | Remove one hidden layer |59.2% |53.8% |54.9% |Test accuracy improved→ simpler model can generalize better |
| Exp 3      | Add a third hidden layer Dense(32) |57.9%|54.6%|51.0% |Train went slightly down, Test decreased → extra layer didn’t help|
| Exp 4      | Activation: relu → tanh |41.5%|39.3%|39.4%|Very poor performance → tanh not suitable here. |
| Exp 5      | Epochs: 10 → 20 | 64.3%| 50.5%|50.8% |creating a large gap that clearly indicates overfitting |
| Exp 6      | Learning rate: 0.001 → 0.01 |52.9% |51.7%|50.1%|High learning rate → model jumps over optimum → unstable training → low accuracy|

**Questions:**
1. Which configuration gives the best test accuracy?
Exp 2 (Remove one hidden layer) → Test Acc = 54.9%
2. In which experiment do you observe overfitting? How can you see it on the curves?
Exp 5 (Epochs 10 → 20)
Overfitting clues:
Train Accuracy high (64.3%)
Validation Accuracy much lower (50.5%)
On the curve: Train curve rises above Val curve → big gap = overfitting
3. What is the effect of increasing the number of neurons?
More neurons don’t always improve performance.
In Exp 1, fewer neurons slightly reduced accuracy → network has less capacity.
Too many neurons can overfit or complicate learning
4. Why does a very high learning rate sometimes give poor results?
Train accuracy unstable becauce large steps during optimization → might overshoot minima → model doesn’t converge well

---
*TP 1 — Artificial Neural Networks*